# T-KGAT gộp, không chạy baseline

Input: `tkgat-joint-code.zip` và thư mục **processed/lastfm-3m-k10** đã được Kaggle giải nén.
Không cần thư mục export, raw interactions hay Freebase dump.

Giữ toàn bộ dữ liệu, gamma `{1,3,10}` và 5 seed. Tham số cố định nằm trong `configs/model/tkgat_joint.yaml`.
**Trainer vẫn dùng full graph:** preflight sẽ chặn nếu chỉ riêng graph/embedding đã vượt bộ nhớ. Notebook này không biến trainer thành streaming.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

INPUT = Path("/kaggle/input")
CODE = Path("/kaggle/working/tkgat-joint-code")
OUTPUT = Path("/kaggle/working/tkgat-joint")
CODE.mkdir(parents=True, exist_ok=True)

archives = list(INPUT.rglob("tkgat-joint-code.zip"))
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            p = Path(name)
            if p.is_absolute() or ".." in p.parts or "\\" in name:
                raise ValueError(f"Unexpected ZIP member: {name}")
        archive.extractall(CODE)
else:
    manifests = list(INPUT.rglob("SOURCE_SHA256.json"))
    if len(manifests) != 1:
        raise RuntimeError("Add exactly one new tkgat-joint-code ZIP/dataset as Input.")
    source = manifests[0].parent
    manifest = json.loads(manifests[0].read_text())
    for name in [*manifest, "SOURCE_SHA256.json"]:
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts or "\\" in name:
            raise ValueError(f"Unexpected source path: {name}")
        target = CODE / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / relative, target)

manifest = json.loads((CODE / "SOURCE_SHA256.json").read_text())
for name, expected in manifest.items():
    if hashlib.sha256((CODE / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Source hash mismatch: {name}")

datasets = [p.parent for p in INPUT.rglob("meta.json")
            if p.parent.name in {"lastfm-3m-k10", "lastfm-raw"}
            and (p.parent / "train.csv").is_file()
            and (p.parent / "kg/default/kg_meta.json").is_file()]
if len(datasets) != 1:
    raise RuntimeError("Add exactly one extracted processed/lastfm-3m-k10 directory as Input; CSV files must be visible.")
PROCESSED = datasets[0]
print("Processed input:", PROCESSED)
print("Output:", OUTPUT)
print("No baseline or data generation will run.")


In [ ]:
# The standard Kaggle Python image should already have these. Preserve its CUDA PyTorch.
import numpy, pandas, scipy, torch, yaml

def run_tkgat(*extra):
    command = [sys.executable, "-u", str(CODE / "scripts/kaggle_tkgat_joint.py"),
               "--processed", str(PROCESSED), "--out", str(OUTPUT), *map(str, extra)]
    env = {**os.environ, "PYTHONPATH": str(CODE / "src")}
    subprocess.run(command, env=env, check=True)

run_tkgat("--check-only")
report = json.loads((OUTPUT / "memory-report.json").read_text())
if report["blockers"]:
    print("\nSTOP: full training does not fit this machine. No interactions were sampled.")
    print("\n".join(report["blockers"]))
else:
    print("Lower-bound check passed; actual peak RAM/VRAM is higher and is not guaranteed to fit.")


In [ ]:
RUN_TRAINING = False  # Set True only after reviewing the memory report.
MAX_NEW_RUNS = 1      # One completed gamma/seed fit per call; rerun this cell to continue.
RESTORE_FROM = None   # Saved previous output directory, e.g. /kaggle/input/<previous-notebook>/tkgat-joint

if RUN_TRAINING:
    if report["blockers"]:
        raise RuntimeError("Preflight blocked training: full graph requires larger hardware or a different memory backend.")
    extra = ["--max-new-runs", str(MAX_NEW_RUNS)]
    if RESTORE_FROM and not (OUTPUT / "registry.csv").exists() and not (OUTPUT / "runs").exists():
        extra += ["--restore", RESTORE_FROM]
    run_tkgat(*extra)
else:
    print("Training is off. No models were trained.")


## Lưu và tiếp tục

Lưu phiên bản notebook **cùng output**, giữ toàn bộ `/kaggle/working/tkgat-joint/` (registry, runs/checkpoint, identity, log).
Phiên sau thêm output đã lưu làm Input, đặt `RESTORE_FROM` tới thư mục `tkgat-joint`, rồi chạy lại notebook.
Trong cùng phiên chỉ cần chạy lại ô train. Một lượt bị ngắt giữa epoch sẽ chạy lại từ đầu.
Chỉ sau khi đủ **3 gamma × 5 seed**, runner chọn gamma trên validation rồi chấm test của gamma đã chọn.
Không dùng checkpoint Stage 1/2 hoặc baseline cũ.
